# Workshop 4, Book inspired exercises - comparing two populations

## Toolbox

The functions you need for Workshop 4. The lecture notebook shows them in use. The exercises give summary statistics, not raw data, so you compute the test statistic from the formulas below and use `scipy` for critical values and p-values.

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std(ddof=1)`, `.var(ddof=1)` | Mean, standard deviation and variance of a column, dividing by $n-1$ |
| `len(x)`, `x.sum()` | Number of observations, number of 1s in a 0/1 column |
| `df.loc[df["col"] == "value", "other"]` | The column `other` for the rows where `col` has a given value |
| `df["a"] - df["b"]` | The difference in each row: one column of changes for paired data |
| `np.sqrt(x)` | Square root (for standard errors) |
| `stats.t.ppf(1 - alpha, df=...)`, `stats.f.ppf(1 - alpha, nu1, nu2)` | One-sided critical values: $\alpha$ above them |
| `stats.t.ppf(1 - alpha / 2, df=...)`, `stats.norm.ppf(1 - alpha / 2)`, `stats.f.ppf(1 - alpha / 2, nu1, nu2)` | Two-sided critical values: $\alpha/2$ above them |
| `stats.t.sf(t, df=...)`, `stats.norm.sf(z)`, `stats.f.sf(F, nu1, nu2)` | Area above a value: the $p$-value when $H_1$ has $>$. `cdf` gives the area below |
| `2 * stats.t.sf(abs(t), df=...)`, `2 * stats.norm.sf(abs(z))` | Two-sided $p$-value |
| `stats.ttest_rel(x, y, alternative=...)` | Paired $t$ test of x − y, as a check |
| `stats.ttest_ind(x1, x2, equal_var=False, alternative=...)` | Welch's $t$ test for two independent means, as a check. Without `equal_var=False` it is the pooled test |
| `stats.skew(x)`, `plt.hist(x, bins=15)` | Skewness and histogram, to check normality |

### Test statistics

| Test | Statistic | Distribution under $H_0$ |
|---|---|---|
| Paired $t$ | $t = \bar d / (s_d/\sqrt{n})$ | $t$ with $n-1$ df |
| Two means, variances known | $z = (\bar x_1 - \bar x_2)/\sqrt{\sigma_1^2/n_1 + \sigma_2^2/n_2}$ | standard normal |
| Two means, pooled | $t = (\bar x_1 - \bar x_2)/\big(s_p\sqrt{1/n_1 + 1/n_2}\big)$ | $t$ with $n_1+n_2-2$ df |
| Two means, Welch | $t = (\bar x_1 - \bar x_2)/\sqrt{s_1^2/n_1 + s_2^2/n_2}$ | $t$ with Welch $v$ df |
| Two proportions | $z = (\hat p_1 - \hat p_2)/\sqrt{\hat p_0(1-\hat p_0)(1/n_1+1/n_2)}$ | approx. standard normal |
| Two variances | $F = s_1^2/s_2^2$ (larger variance on top for one-sided) | $F$ with $n_1-1$, $n_2-1$ df |

For any function, and for code an LLM writes: check what it computes before you use the result.

<details>
<summary>Toolbox from Workshop 3 (one-sample tests)</summary>

| Function | What it does |
|---|---|
| `np.array([...])` | Turn a list of numbers into a sample |
| `len(x)`, `x.mean()` | Sample size $n$ and sample mean $\bar{x}$ |
| `x.std(ddof=1)`, `x.var(ddof=1)` | Sample SD $s$ and variance $s^2$ (divide by $n-1$) |
| `norm.ppf(1 - alpha)`, `norm.ppf(alpha)`, `norm.ppf(1 - alpha/2)` | Critical $z$: upper-tailed, lower-tailed, two-sided |
| `norm.cdf(z)`, `norm.sf(z)` | Lower- and upper-tailed $p$-value for a $z$-test |
| `stats.t.ppf(1 - alpha, df=n-1)`, `stats.t.ppf(alpha, df=n-1)`, `stats.t.ppf(1 - alpha/2, df=n-1)` | Critical $t$: upper-tailed, lower-tailed, two-sided |
| `stats.t.cdf(t, df=n-1)`, `stats.t.sf(t, df=n-1)` | Lower- and upper-tailed $p$-value for a $t$-test |
| `stats.chi2.ppf(1 - alpha, df=n-1)`, `stats.chi2.sf(chi2, df=n-1)` | Critical value and upper-tailed $p$-value for a variance test |

One-sample statistics:

$$
z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}, \qquad
t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}\ (df = n-1), \qquad
z = \frac{\hat p - P_0}{\sqrt{P_0(1-P_0)/n}}
$$

</details>

<details>
<summary>Toolbox from Workshop 2 (confidence intervals)</summary>

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std(ddof=1)`, `.var(ddof=1)` | Mean, standard deviation and variance of a column, dividing by n − 1 |
| `len(x)` | Number of observations |
| `df.loc[df["col"] == "value", "other"]` | The column `other` for the rows where `col` has a given value |
| `(df["col"] > value).mean()` | Share of rows where the column is above a value |
| `np.sqrt(x)` | Square root |
| `stats.norm.ppf(q)` | Value with probability q below it in the standard normal: $z_{\alpha/2}$ is `stats.norm.ppf(1 - alpha / 2)` |
| `stats.t.ppf(q, df=n - 1)` | The same for the t distribution with n − 1 degrees of freedom |
| `stats.chi2.ppf(q, df=n - 1)` | The same for the chi-square distribution with n − 1 degrees of freedom |
| `stats.skew(x)` | Skewness |
| `np.ceil(x)` | Round up to the next whole number |
| `plt.hist(x, bins=20)` | Histogram |
| `rng = np.random.default_rng(seed)`, `rng.choice(x, size=n, replace=False)` | Random sample of n values |

</details>

<details>
<summary>Toolbox from Workshop 1 (descriptive statistics, sampling, distributions)</summary>

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std()`, `.var()` | Mean, standard deviation and variance of a column (pandas divides by n − 1) |
| `np.std(x)`, `np.var(x)` | Standard deviation and variance (numpy divides by n unless you add `ddof=1`) |
| `df.groupby("col")["other"].agg(["mean", "std"])` | Statistics of one column for each group in another |
| `df[df["col"] == "value"]` | The rows where a column has a given value |
| `x.head(5)` | The first five values |
| `rng = np.random.default_rng(seed)` | A random number generator; the seed makes the results repeatable |
| `rng.choice(x, size=n, replace=False)` | A simple random sample of n values from x |
| `np.array(a_list)` | Turns a list into an array, so you can compute with it |
| `np.mean(a > value)` | The share of values in an array that are above a value |
| `np.sqrt(x)` | Square root |
| `stats.skew(x)` | Skewness |
| `stats.norm.cdf(x, loc, scale)`, `.sf`, `.ppf` | Normal distribution: probability below x, probability above x, value with a given probability below it |
| `stats.chi2.cdf(x, df)`, `.sf`, `.ppf` | The same for the chi-square distribution |
| `stats.binom.sf(k, n, p)` | Probability of more than k successes in n trials |
| `plt.hist(x, bins=30)` | Histogram |

For simulations: draw a sample inside a `for` loop, compute a statistic, and append it to a list.

</details>

For every task: state $H_0$ and $H_1$, compute the test statistic, compare it with the critical value, report the $p$-value and explain the decision in context.

## Setup

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt

# Warm-up (one-sample review)

Before comparing **two** populations, these three warm-ups review the **one-sample** tests from Chapter 9. They are rewritten (new situations and new numbers) from Chapter 9 book exercises that were available for Workshop Lecture 3 but were **not** used in it:

| Warm-up | Test | Book template |
|---|---|---|
| A | Mean, known $\sigma$ ($z$-test) | 9.12 |
| B | Mean, $\sigma$ unknown ($t$-test) | 9.19 |
| C | Proportion, large sample ($z$-test) | 9.30 |

The three statistics are

$$
z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}, \qquad t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}\ (df = n-1), \qquad z = \frac{\hat{p} - P_0}{\sqrt{P_0(1-P_0)/n}}
$$

## Warm-up

## Warm-up A: Courier delivery times (known $\sigma$, $z$-test)

A courier company claims that same-day parcels are delivered in **at most 4.5 hours** on average. Delivery times are normally distributed with known standard deviation $\sigma = 0.9$ hours. A random sample of $n = 36$ parcels has a mean delivery time of $\bar{x} = 4.82$ hours. Test at $\alpha = 0.05$ whether the mean delivery time exceeds 4.5 hours.

$$
H_0: \mu \le 4.5 \qquad \text{versus} \qquad H_1: \mu > 4.5
$$

<details>
<summary>Hint 1</summary>

σ is known, so this is a $z$-test. Compute $SE = \sigma/\sqrt{n}$, then $z = (\bar{x}-\mu_0)/SE$. The test is upper-tailed.
</details>

<details>
<summary>Hint 2</summary>

`norm.ppf(1 - alpha)` for the critical value, `norm.sf(z)` for the $p$-value, and `np.sqrt`.
</details>

**Your answer:**


## Warm-up B: Usefulness of career mentoring ($\sigma$ unknown, $t$-test)

A university careers office asks a random sample of $n = 48$ recent graduates to rate, on a scale from 1 (not useful) to 5 (very useful), how useful career mentoring was. The sample mean rating is $\bar{x} = 3.48$ with sample standard deviation $s = 0.82$. Test at $\alpha = 0.01$ whether the population mean rating is greater than 3.0.

$$
H_0: \mu \le 3.0 \qquad \text{versus} \qquad H_1: \mu > 3.0
$$

<details>
<summary>Hint 1</summary>

σ is unknown, so use $t = (\bar{x}-\mu_0)/(s/\sqrt{n})$ with $n-1$ degrees of freedom. The test is upper-tailed, at $\alpha = 0.01$.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(1 - alpha, df=n-1)` for the critical value and `stats.t.sf(t, df=n-1)` for the $p$-value.
</details>

**Your answer:**


## Warm-up C: Abandoned online carts and shipping cost (proportion $z$-test)

In a random sample of $n = 280$ online shoppers who abandoned a cart, 98 said that shipping cost was the main reason. Test at $\alpha = 0.05$ whether **more than 30%** of all such shoppers abandon their cart mainly because of shipping cost.

$$
H_0: P \le 0.30 \qquad \text{versus} \qquad H_1: P > 0.30
$$

Under $H_0$ the standard error uses $P_0 = 0.30$: $\;SE = \sqrt{P_0(1-P_0)/n}$. The large-sample condition holds since $nP_0(1-P_0) = 280(0.30)(0.70) = 58.8 > 5$.

<details>
<summary>Hint 1</summary>

Compute $\hat p = x/n$, then $z = (\hat p - P_0)/\sqrt{P_0(1-P_0)/n}$. The standard error uses $P_0$, not $\hat p$. Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

`norm.ppf(1 - alpha)` and `norm.sf(z)`. Also compare your decision at $\alpha = 0.01$.
</details>

**Your answer:**


## Part 1 — Paired $t$-test for a mean difference

With **paired** observations we work with the differences $d_i = x_{1i} - x_{2i}$. If the differences are approximately normal and $\sigma_d$ is unknown, the test statistic for $H_0: \mu_1 - \mu_2 = D_0$ is

$$
t = \frac{\bar{d} - D_0}{s_d/\sqrt{n}}, \qquad df = n - 1
$$

where $\bar{d} = \bar{x}_1 - \bar{x}_2$ is the sample mean difference, $s_d$ is the sample standard deviation of the differences and $s_d/\sqrt{n}$ is the standard error.

| $H_1$ | Reject $H_0$ if | $p$-value |
|---|---|---|
| $\mu_1-\mu_2 > 0$ | $t > t_{n-1,\alpha}$ | $P(T > t)$ → `stats.t.sf(t, df)` |
| $\mu_1-\mu_2 < 0$ | $t < -t_{n-1,\alpha}$ | $P(T < t)$ → `stats.t.cdf(t, df)` |
| $\mu_1-\mu_2 \neq 0$ | $|t| > t_{n-1,\alpha/2}$ | $2P(T > |t|)$ |

## Problem

You have been asked to determine if two different production processes have different mean numbers of units produced per hour. Process 1 has mean $\mu_1$ and process 2 has mean $\mu_2$. The hypotheses are:

$$
H_0: \mu_1 - \mu_2 = 0 \qquad \text{versus} \qquad H_1: \mu_1 - \mu_2 > 0
$$

Using a random sample of $n = 25$ paired observations, the sample means are $\bar{x}_1 = 50$ and $\bar{x}_2 = 60$. Can you reject the null hypothesis using a probability of Type I error $\alpha = 0.05$ in each case?

a. The sample standard deviation of the difference is $s_d = 20$.

b. The sample standard deviation of the difference is $s_d = 25$.

<details>
<summary>Hint 1</summary>

Work with the mean difference $\bar d = \bar x_1 - \bar x_2$ and $df = n-1 = 24$. Look at the **sign** of $\bar d$ and compare it with the direction of $H_1$ before you interpret anything.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(1 - alpha, df=24)` and `stats.t.sf(t, df=24)`. For extra insight, also try the lower-tailed (`stats.t.cdf`) and two-sided versions and see how the decision changes.
</details>

### Part (a): $s_d = 20$

**Your answer:**


### Part (b): $s_d = 25$

**Your answer:**


---
# Part 2: Independent samples with known variances ($z$-test)

When the two samples are **independent** and the population variances $\sigma_1^2$ and $\sigma_2^2$ are **known**, the test statistic for $H_0: \mu_1 - \mu_2 = D_0$ is

$$
z = \frac{(\bar{x}_1 - \bar{x}_2) - D_0}{\sqrt{\dfrac{\sigma_1^2}{n_1} + \dfrac{\sigma_2^2}{n_2}}}
$$

For the upper-tailed alternative $H_1: \mu_1 - \mu_2 > 0$, reject $H_0$ if $z > z_\alpha$. The $p$-value is $P(Z > z)$, computed with `norm.sf(z)`.

## Problem

You have been asked to determine if two different production processes have different mean numbers of units produced per hour. Process 1 has mean $\mu_1$ and process 2 has mean $\mu_2$. The hypotheses are:

$$
H_0: \mu_1 - \mu_2 = 0 \qquad \text{versus} \qquad H_1: \mu_1 - \mu_2 > 0
$$

Use a random sample of $n_1 = 25$ observations from process 1 and $n_2 = 28$ observations from process 2. The known variance for process 1 is $\sigma_1^2 = 900$ and the known variance for process 2 is $\sigma_2^2 = 1600$. Can you reject the null hypothesis using a probability of Type I error $\alpha = 0.05$ in each case?

a. The sample means are $\bar{x}_1 = 50$ and $\bar{x}_2 = 60$.

b. The difference in sample means is $\bar{x}_1 - \bar{x}_2 = 20$.

<details>
<summary>Hint 1</summary>

Known variances, independent samples: $SE = \sqrt{\sigma_1^2/n_1 + \sigma_2^2/n_2}$ and $z = ((\bar x_1-\bar x_2) - D_0)/SE$. Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

`norm.ppf(1 - alpha)`, `norm.sf(z)` and `np.sqrt`. In (a) compute $\bar x_1 - \bar x_2$ first and check its sign.
</details>

### Part 2(a): $\bar{x}_1 = 50$, $\bar{x}_2 = 60$

**Your answer:**


### Part 2(b): $\bar{x}_1 - \bar{x}_2 = 20$

**Your answer:**


---
# Part 3: Independent samples, unknown but equal variances (pooled $t$-test)

When the population variances are unknown but assumed **equal**, we pool the two sample variances:

$$
s_p^2 = \frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}, \qquad
t = \frac{(\bar{x}_1 - \bar{x}_2) - D_0}{s_p\sqrt{\dfrac{1}{n_1} + \dfrac{1}{n_2}}}, \qquad df = n_1 + n_2 - 2
$$

For an upper-tailed test, reject $H_0$ if $t > t_{n_1+n_2-2,\alpha}$.

## Problem

You have been asked to determine if two different production processes have different mean numbers of units produced per hour. Process 1 has mean $\mu_1$ and process 2 has mean $\mu_2$. The hypotheses are:

$$
H_0: \mu_1 - \mu_2 \le 0 \qquad \text{versus} \qquad H_1: \mu_1 - \mu_2 > 0
$$

The population variances are unknown but assumed equal. A random sample of $n_1 = 25$ observations from process 1 gives $\bar{x}_1 = 56$ and $s_1 = 30$; a random sample of $n_2 = 36$ observations from process 2 gives $\bar{x}_2 = 50$ and $s_2 = 28$. Can you reject the null hypothesis using a probability of Type I error $\alpha = 0.05$?

<details>
<summary>Hint 1</summary>

Compute the pooled variance $s_p^2$ first, then $SE = s_p\sqrt{1/n_1 + 1/n_2}$ and $t$ with $df = n_1+n_2-2$. Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(1 - alpha, df=n1 + n2 - 2)` and `stats.t.sf(t, df=n1 + n2 - 2)`.
</details>

**Your answer:**


---
# Parts 4–5: Independent samples, unknown and possibly unequal variances (Welch $t$-test)

When the population variances are unknown and **not** assumed equal, use

$$
t = \frac{(\bar{x}_1 - \bar{x}_2) - D_0}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}}
$$

with the Welch–Satterthwaite degrees of freedom

$$
v = \frac{\left(\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}\right)^2}{\dfrac{(s_1^2/n_1)^2}{n_1 - 1} + \dfrac{(s_2^2/n_2)^2}{n_2 - 1}}
$$

For an upper-tailed test, reject $H_0$ if $t > t_{v,\alpha}$; the $p$-value is `stats.t.sf(t, df=v)`. Neither problem states a significance level, so we use $\alpha = 0.05$ and also report the $p$-value. The pooled $t$-test from Part 3 is shown as an optional check.

## Part 4: Attitudes toward minorities as managers

A screening procedure was designed to measure attitudes toward minorities as managers. High scores indicate negative attitudes and low scores indicate positive attitudes. Independent random samples were taken of $n_M = 151$ male financial analysts and $n_F = 108$ female financial analysts. For the male group the sample mean and standard deviation of scores were $\bar{x}_M = 85.8$ and $s_M = 19.13$; for the female group they were $\bar{x}_F = 71.5$ and $s_F = 12.2$.

Test the null hypothesis that the two population means are equal against the alternative that the true mean score is higher for male than for female financial analysts:

$$
H_0: \mu_M - \mu_F = 0 \qquad \text{versus} \qquad H_1: \mu_M - \mu_F > 0
$$

<details>
<summary>Hint 1</summary>

Unequal variances: $t = (\bar x_M - \bar x_F)/\sqrt{s_M^2/n_M + s_F^2/n_F}$, with the Welch–Satterthwaite degrees of freedom $v$ (not an integer). Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

Compute $a_M = s_M^2/n_M$ and $a_F = s_F^2/n_F$ first, then $v = (a_M+a_F)^2/\big(a_M^2/(n_M-1) + a_F^2/(n_F-1)\big)$. Use `stats.t.ppf(1 - alpha, df=v)` and `stats.t.sf(t, df=v)`.
</details>

**Your answer:**


## Part 5: Job changes — entrepreneurs vs corporate managers

For a random sample of $n_E = 125$ British entrepreneurs, the mean number of job changes was $\bar{x}_E = 1.91$ and the sample standard deviation was $s_E = 1.32$. For an independent random sample of $n_M = 86$ British corporate managers, the mean number of job changes was $\bar{x}_M = 0.21$ and the sample standard deviation was $s_M = 0.53$.

Test the null hypothesis that the population means are equal against the alternative that the mean number of job changes is higher for British entrepreneurs than for British corporate managers:

$$
H_0: \mu_E - \mu_M = 0 \qquad \text{versus} \qquad H_1: \mu_E - \mu_M > 0
$$

<details>
<summary>Hint 1</summary>

Same Welch setup as Part 4. Notice how different the two sample standard deviations are, and think about why that matters for the choice of test.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(1 - alpha, df=v)` and `stats.t.sf(t, df=v)`. Optionally compare with the pooled $t$-test from Part 3.
</details>

**Your answer:**


---
# Parts 6–7: Difference between two population proportions ($z$-test)

For independent large samples, to test $H_0: P_1 - P_2 = 0$ we use the **pooled** sample proportion (because under $H_0$ the two proportions are equal):

$$
\hat{p}_0 = \frac{n_1\hat{p}_1 + n_2\hat{p}_2}{n_1 + n_2}, \qquad
z = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\hat{p}_0(1 - \hat{p}_0)\left(\dfrac{1}{n_1} + \dfrac{1}{n_2}\right)}}
$$

| $H_1$ | Reject $H_0$ if | $p$-value |
|---|---|---|
| $P_1 - P_2 > 0$ | $z > z_\alpha$ | `norm.sf(z)` |
| $P_1 - P_2 \neq 0$ | $|z| > z_{\alpha/2}$ | `2 * norm.sf(abs(z))` |

## Part 6: Economic optimism — United States vs Great Britain

Random samples of $n_{US} = 900$ people in the United States and $n_{GB} = 900$ people in Great Britain indicated that $\hat{p}_{US} = 0.60$ of the people in the United States were positive about the future economy, whereas $\hat{p}_{GB} = 0.66$ of the people in Great Britain were positive about the future economy. Does this provide strong evidence that the people in Great Britain are more optimistic about the economy?

$$
H_0: P_{GB} - P_{US} = 0 \qquad \text{versus} \qquad H_1: P_{GB} - P_{US} > 0
$$

Use $\alpha = 0.05$ and report the $p$-value.

<details>
<summary>Hint 1</summary>

Compute the pooled proportion $\hat p_0 = (n_1\hat p_1 + n_2\hat p_2)/(n_1+n_2)$, then $z = (\hat p_{GB}-\hat p_{US})/\sqrt{\hat p_0(1-\hat p_0)(1/n_1+1/n_2)}$. Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

`norm.ppf(1 - alpha)`, `norm.sf(z)` and `np.sqrt`.
</details>

**Your answer:**


## Part 7 (10.16): Attitudes toward world trade — country A vs country B

A random sample of $n_A = 1556$ people in country A were asked to respond to this statement: *Increased world trade can increase our per capita prosperity.* Of these sample members, $\hat{p}_A = 0.384$ (38.4%) agreed with the statement. When the same statement was presented to a random sample of $n_B = 1108$ people in country B, $\hat{p}_B = 0.520$ (52.0%) agreed. Test the null hypothesis that the population proportions agreeing with this statement were the same in the two countries against the two-sided alternative:

$$
H_0: P_A - P_B = 0 \qquad \text{versus} \qquad H_1: P_A - P_B \neq 0
$$

Use $\alpha = 0.05$ and report the $p$-value.

<details>
<summary>Hint 1</summary>

Same pooled-proportion $z$-test, but the alternative is **two-sided**: reject if $|z| > z_{\alpha/2}$.
</details>

<details>
<summary>Hint 2</summary>

`norm.ppf(1 - alpha/2)` for the critical value and `2 * norm.sf(abs(z))` for the $p$-value.
</details>

**Your answer:**


---
# Parts 8–9: Equality of two population variances ($F$-test)

For independent random samples from two **normal** populations, to test $H_0: \sigma_x^2 = \sigma_y^2$ against $H_1: \sigma_x^2 > \sigma_y^2$ we use

$$
F = \frac{s_x^2}{s_y^2}, \qquad \text{with } (n_x - 1,\; n_y - 1) \text{ degrees of freedom}
$$

where the variance under the alternative's "larger" side goes in the **numerator**. Reject $H_0$ if $F > F_{n_x-1,\,n_y-1,\,\alpha}$; the $p$-value is $P(F_{n_x-1,n_y-1} > F)$, computed with `stats.f.sf(F, dfn, dfd)`.

## Part 8: Variability of tax-filing judgments by expertise

It is hypothesized that the more expert a group of people examining personal income tax filings, the more variable the judgments will be about the accuracy. Independent random samples, each of 30 individuals, were chosen from groups with different levels of expertise. The low-expertise group consisted of people who had just completed their first intermediate accounting course. Members of the high-expertise group had completed undergraduate studies and were employed by reputable CPA firms. The sample members were asked to judge the accuracy of personal income tax filings. For the low-expertise group the sample variance was $s_L^2 = 451.770$, whereas for the high-expertise group it was $s_H^2 = 1614.208$. Test the null hypothesis that the two population variances are equal against the alternative that the true variance is higher for the high-expertise group:

$$
H_0: \sigma_H^2 = \sigma_L^2 \qquad \text{versus} \qquad H_1: \sigma_H^2 > \sigma_L^2
$$

No significance level is given, so we use $\alpha = 0.05$ (and also check $\alpha = 0.01$) and report the $p$-value. We assume both populations are normal.

<details>
<summary>Hint 1</summary>

Put the variance of the group named in $H_1$ (high expertise) in the numerator: $F = s_H^2/s_L^2$ with $(n_H-1,\,n_L-1)$ degrees of freedom. Upper-tailed test.
</details>

<details>
<summary>Hint 2</summary>

`stats.f.ppf(1 - alpha, dfn, dfd)` and `stats.f.sf(F, dfn, dfd)`. Repeat the critical value for $\alpha = 0.01$.
</details>

**Your answer:**


## Part 9 (10.24): Variability of ship sales — price competition vs duopoly

It is hypothesized that the total sales of a corporation should vary more in an industry with active price competition than in one with duopoly and tacit collusion. In a study of the merchant ship production industry it was found that in $n_C = 4$ years of active price competition, the variance of company A's total sales was $s_C^2 = 114.09$. In the following $n_D = 7$ years, during which there was duopoly and tacit collusion, this variance was $s_D^2 = 16.08$. Assume that the data can be regarded as an independent random sample from two normal distributions. Test, at the 5% level, the null hypothesis that the two population variances are equal against the alternative that the variance of total sales is higher in years of active price competition:

$$
H_0: \sigma_C^2 = \sigma_D^2 \qquad \text{versus} \qquad H_1: \sigma_C^2 > \sigma_D^2
$$

<details>
<summary>Hint 1</summary>

Same $F$-test: $F = s_C^2/s_D^2$ with $(n_C-1,\,n_D-1)$ degrees of freedom. With so few observations, think about how much you trust the result.
</details>

<details>
<summary>Hint 2</summary>

`stats.f.ppf(1 - alpha, dfn, dfd)` and `stats.f.sf(F, dfn, dfd)`.
</details>

**Your answer:**
